In [4]:
import pandas as pd 
import numpy as np
from datetime import datetime, date

In [5]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v1/metadata_index/data_assets


In [13]:
aggregate = [
  {
    "$match": {
      "data_description.project_name": "V1 Deep Dive", 
      "name": {"$regex": "filtered"},
      "location": {"$regex": "aind-open-data"}, 
      "processing.data_processes": {
        "$elemMatch": {
          "name": "Filter NWB values",
        }
      }
  },
  }
  ,
  {
    "$project": {
      "name": 1, 
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.subject_details.genotype", 
      "date_of_birth": "$subject.subject_details.date_of_birth", 
      "sex": "$subject.subject_details.sex", 
      "session_start_time": "$acquisition.acquisition_start_time",
      "session_end_time": "$acquisition.acquisition_end_time",
      "project_name": "$data_description.project_name", 
      "modality": "$data_description.modalities.name",
      "column": { "$arrayElemAt": ["$data_description.tags", 0] },
      "volume": { "$arrayElemAt": ["$data_description.tags", 1] }
    }
  },
]
    
records = docdb_api_client.aggregate_docdb_records(
    pipeline = aggregate,
)

In [15]:
df = pd.DataFrame(records)

df['session_date'] = df.apply(lambda x: datetime.fromisoformat(x['session_start_time']).date(), axis=1)
df['session_start_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_start_time']).time(), axis=1)
df['session_end_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_end_time']).time(), axis=1)
df['date_of_birth'] = df.apply(lambda x: datetime.strptime(x['date_of_birth'], '%Y-%m-%d').date(), axis=1)
df['age'] = df.apply(lambda x: (x['session_date'] - x['date_of_birth']).days, axis=1)

df['column'] = df.apply(lambda x: int(x['column'].split(' ')[-1]), axis=1)
df['volume'] = df.apply(lambda x: int(x['volume'].split(' ')[-1]), axis=1)

df['golden_mouse'] = False
df.loc[df.subject_id=='409828', 'golden_mouse'] = True

order = ['project_name','_id','name','subject_id','golden_mouse','genotype','date_of_birth','age', 'sex','modality',
         'session_date','session_start_time', 'session_end_time','column','volume']

df = df[order].sort_values(by=['subject_id', 'column', 'volume'])
df

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,age,sex,modality,session_date,session_start_time,session_end_time,column,volume
69,V1 Deep Dive,2b93bc7c-c2dc-4347-9279-80f5c3e7ddb0,409828_2018-12-12_12-24-18_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,162,Male,"[Planar optical physiology, Behavior videos]",2018-12-12,12:24:18.963730,13:24:27.772570,1,1
70,V1 Deep Dive,99fb8537-40f8-4960-ad02-dd723124e7d6,409828_2018-12-13_13-21-39_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,163,Male,"[Planar optical physiology, Behavior videos]",2018-12-13,13:21:39.888000,14:21:12.698880,1,2
1,V1 Deep Dive,b1e7cf7c-bb52-42c9-9dde-54e7594b3ae2,409828_2018-12-13_15-10-05_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,163,Male,"[Planar optical physiology, Behavior videos]",2018-12-13,15:10:05.562960,16:09:36.222020,1,3
2,V1 Deep Dive,2cb93697-c954-4835-9c17-6deed1cf1fb5,409828_2018-12-14_13-14-42_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,164,Male,"[Planar optical physiology, Behavior videos]",2018-12-14,13:14:42.634500,14:15:00.029230,1,4
3,V1 Deep Dive,5aac054c-c0ce-4c19-94e9-8135f047fa62,409828_2018-12-14_14-47-35_filtered_2026-04-09...,409828,True,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-07-03,164,Male,"[Planar optical physiology, Behavior videos]",2018-12-14,14:47:35.097180,15:46:50.098970,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,V1 Deep Dive,89a2b07f-a080-44a1-8262-f25b39563c1d,438833_2019-03-21_14-08-14_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,110,Male,"[Planar optical physiology, Behavior videos]",2019-03-21,14:08:14.261290,15:10:53.254690,5,1
96,V1 Deep Dive,de0aad16-7644-443a-83dc-029bc7342354,438833_2019-03-27_11-33-05_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,116,Male,"[Planar optical physiology, Behavior videos]",2019-03-27,11:33:05.464370,12:31:50.157580,5,2
27,V1 Deep Dive,40fd8a56-ce65-42fb-815f-f39a46e4ebb9,438833_2019-03-27_14-24-07_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,116,Male,"[Planar optical physiology, Behavior videos]",2019-03-27,14:24:07.041320,15:23:46.589260,5,3
30,V1 Deep Dive,bddf33d3-17c7-4095-9649-92e5b712a866,438833_2019-04-01_14-39-42_filtered_2026-04-09...,438833,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-12-01,121,Male,"[Planar optical physiology, Behavior videos]",2019-04-01,14:39:42.230830,15:39:10.801820,5,4


In [16]:
df.to_csv('/data/metadata/V1DD_metadata.csv', index= False)